In [ ]:
# Imports and Setup
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# This notebook is in <project_root>/causalgym/test/
# cartpole_wind_pch.py is assumed to be in <project_root>

# Assuming CWD is the notebook's directory when run.
current_notebook_actual_dir = os.getcwd() # Expected: C:/Users/Matthew/Documents/causal2/causalgym/test/

# Path to the project root (two levels up from causalgym/test/)
project_root_from_nb = os.path.abspath(os.path.join(current_notebook_actual_dir, "..", ".."))
if project_root_from_nb not in sys.path:
    sys.path.insert(0, project_root_from_nb) 
    print(f"Added to sys.path (for cartpole_wind_pch.py): {project_root_from_nb}")

try:
    from cartpole_wind_pch import CartPoleWindPCH
except ImportError as e:
    print(f"ImportError: {e}. Please ensure cartpole_wind_pch.py is in the project root: {project_root_from_nb}")
    print(f"Current sys.path: {sys.path}")

# Helper function to display frames
def display_frames(frames, title="Frames"):
    if not frames:
        print(f"{title}: No frames to display.")
        return
    fig, axes = plt.subplots(1, len(frames), figsize=(len(frames) * 3, 3))
    if len(frames) == 1:
        axes = [axes] # Make it iterable for single frame
    for i, frame in enumerate(frames):
        axes[i].imshow(frame)
        axes[i].axis('off')
        axes[i].set_title(f"Step {i}")
    fig.suptitle(title)
    plt.show()

print("Setup complete. CartPoleWindPCH and helper function ready.")

In [ ]:
# Test with No Wind
print("\n--- Testing with No Wind ---")

# Instantiate with wind parameters set to zero
env_no_wind = CartPoleWindPCH(wind_mean=0.0, wind_std=0.0, render_mode='rgb_array')
obs, info = env_no_wind.reset(seed=123)
print(f"Initial observation: {obs}")
# Accessing SCM's current_wind. It should be 0.0 as mean and std are 0.
print(f"Initial wind (from SCM internal): {env_no_wind.env.current_wind:.4f}")

frames_no_wind = []
num_steps_test_no_wind = 5

for step in range(num_steps_test_no_wind):
    action = env_no_wind.action_space.sample() # Take random actions
    # Use env.do() for PCH to apply action
    obs, reward, terminated, truncated, info = env_no_wind.do(action)
    frame = env_no_wind.render()
    if frame is not None:
        frames_no_wind.append(frame)
    
    print(f"Step {step+1}: Action={action}, Obs={np.round(obs,2)}, Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env_no_wind.close()
display_frames(frames_no_wind, title="CartPole with No Wind")

In [ ]:
# Test with Default Wind Settings
print("--- Testing with Default Wind ---")

# Default wind_std is 0.01 in CartPoleWindSCM
env_default_wind = CartPoleWindPCH(render_mode='rgb_array')
obs, info = env_default_wind.reset(seed=42)
print(f"Initial observation: {obs}")
print(f"Initial wind (from SCM internal): {env_default_wind.env.current_wind:.4f}") # Accessing SCM's current_wind

frames_default_wind = []
num_steps_test = 5

for step in range(num_steps_test):
    action = env_default_wind.action_space.sample() # Take random actions
    obs, reward, terminated, truncated, info = env_default_wind.do(action)
    frame = env_default_wind.render()
    if frame is not None:
        frames_default_wind.append(frame)
    
    print(f"Step {step+1}: Action={action}, Obs={np.round(obs,2)}, Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env_default_wind.close()
display_frames(frames_default_wind, title="CartPole with Default Wind")

In [ ]:
# Test with Heavy Wind
print("\n--- Testing with Heavy Wind ---")

# Instantiate with a stronger wind effect
# Option 1: Strong consistent wind in one direction
env_heavy_wind = CartPoleWindPCH(wind_mean=0.1, wind_std=0.01, render_mode='rgb_array') 
# Option 2: More variable strong wind (could also try higher std like 0.1 or 0.2)
# env_heavy_wind = CartPoleWindPCH(wind_mean=0.0, wind_std=0.2, render_mode='rgb_array')

obs, info = env_heavy_wind.reset(seed=789)
print(f"Initial observation: {obs}")
print(f"Initial wind (from SCM internal): {env_heavy_wind.env.current_wind:.4f}")

frames_heavy_wind = []
num_steps_test_heavy = 15 # More steps to see the effect

for step in range(num_steps_test_heavy):
    action = env_heavy_wind.action_space.sample() # Try random actions, or a fixed one e.g., action = 0
    obs, reward, terminated, truncated, info = env_heavy_wind.do(action)
    frame = env_heavy_wind.render()
    if frame is not None:
        frames_heavy_wind.append(frame)
    
    print(f"Step {step+1}: Action={action}, Obs={np.round(obs,2)}, Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env_heavy_wind.close()
display_frames(frames_heavy_wind, title=f"CartPole with Heavy Wind (mean={env_heavy_wind.env.wind_mean}, std={env_heavy_wind.env.wind_std})")

In [ ]:
# Test with Initial Pole Angle Offset
print("\n--- Testing with Initial Pole Angle Offset ---")

# Instantiate with a non-zero initial pole angle (e.g., 0.15 radians ~ 8.6 degrees)
# Keep wind minimal for this test to isolate angle effect.
initial_angle_rad = 0.15
env_angled_start = CartPoleWindPCH(
    init_theta_mean=initial_angle_rad, 
    init_theta_std=0.01, # Small std around the mean for consistency, results in angle ~0.15
    wind_mean=0.0,       # Minimal wind
    wind_std=0.001,      # Minimal wind
    render_mode='rgb_array'
)

obs, info = env_angled_start.reset(seed=321)
print(f"Initial observation: {obs}") # obs[2] should be close to initial_angle_rad
print(f"Pole angle from initial obs: {obs[2]:.4f} radians ({np.degrees(obs[2]):.2f} degrees)")
print(f"Initial wind (from SCM internal): {env_angled_start.env.current_wind:.4f}")

frames_angled_start = []
num_steps_test_angled = 30 # Increased from 10 to 30 steps

print(f"Running for {num_steps_test_angled} steps to observe pole fall...")

for step in range(num_steps_test_angled):
    action = env_angled_start.action_space.sample() 
    # To see a more consistent fall, you could try a fixed action, e.g., action = 0 or 1
    # For now, let's stick to random actions.
    obs, reward, terminated, truncated, info = env_angled_start.do(action)
    frame = env_angled_start.render()
    if frame is not None:
        frames_angled_start.append(frame)
    
    print(f"Step {step+1:2d}: Action={action}, Obs={np.round(obs,2)}, PoleAngle={obs[2]:.3f} rad ({np.degrees(obs[2]):.1f} deg), Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
    if terminated or truncated:
        print("Episode finished.")
        break

env_angled_start.close()
display_frames(frames_angled_start, title=f"CartPole with Initial Angle (mean={initial_angle_rad:.2f} rad, {num_steps_test_angled} steps)")